# 16 · Un device riceve un file firmato e lo applica (come un firmware)

Nel notebook 14 il job **consegna** un file al device. Qui guardiamo **cosa fa il device quando lo riceve**: prima di applicarlo **verifica la firma**, poi lo **applica**,
poi lo **prova**; se la prova fallisce **torna alla versione precedente**. Il file, per essere visibile, è un piccolo script di termostato: cambiarlo cambia il comportamento del device.

Il meccanismo, in PowerShell e in locale (nessun Azure):
- il server di rilascio **firma i byte esatti** del file con ECDSA P-384 / SHA-384 (firma IEEE P1363, in esadecimale); sul device c'è solo la **chiave pubblica**;
- il device controlla **hash e firma**; se non tornano, rifiuta e non cambia nulla;
- se la firma è valida, **mette da parte la versione buona**, applica il nuovo file e lo **prova**; se la prova fallisce, **ripristina** la versione buona.

**Da dire.** "Un aggiornamento firmware è pesante e rischioso. Qualunque sia il canale che porta il file, un job o altro, è il device che deve decidere se fidarsi: la **firma** è il confine di sicurezza,
e la **prova con ritorno indietro** copre ciò che la firma non può dire, cioè che il file funzioni."

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## 1 · Le chiavi
La **chiave privata di firma** resta sul server di rilascio. Sul device va **solo la chiave pubblica**.

In [ ]:
$root = Join-Path $out 'sim\06'
if (Test-Path $root) { Remove-Item $root -Recurse -Force }
New-Item -ItemType Directory -Force (Join-Path $root 'server'), (Join-Path $root 'device\payload') | Out-Null

$signKey = [Security.Cryptography.ECDsa]::Create([Security.Cryptography.ECCurve]::CreateFromFriendlyName('nistP384'))
Write-Pem -Path (Join-Path $root 'server\signing-key.pem') -Label 'PRIVATE KEY' -Bytes $signKey.ExportPkcs8PrivateKey()   # NON esce dal server
Write-Pem -Path (Join-Path $root 'device\signing-public.pem') -Label 'PUBLIC KEY' -Bytes $signKey.ExportSubjectPublicKeyInfo()
Get-ChildItem $root -Recurse -File | Select-Object @{n='File';e={$_.FullName.Substring($root.Length+1)}}, Length | Format-Table | Out-String


File                      Length
----                      ------
device\signing-public.pem    215
server\signing-key.pem       306



## 2 · Il server firma, l'agente verifica
`Publish-Payload` firma i **byte esatti** del file (nessun riserializzare). L'agente sul device ha tre difese: **firma** (e hash), **prova** (il payload deve superare un autotest)
e **ritorno alla versione buona** se la prova fallisce.

In [ ]:
# Le funzioni stanno nella libreria (AzureADRIoTHubLib\Payload.ps1). Qui si leggono: il SERVER firma, il DEVICE verifica con la sola chiave pubblica.
foreach ($f in 'Publish-Payload', 'Test-PayloadSignature', 'Install-Payload') { "# ---- $f"; (Get-Command $f).ScriptBlock.ToString().TrimEnd() }
New-Item -ItemType Directory -Force (Join-Path $root 'server\outbox') | Out-Null

# ---- Publish-Payload


    <#
    .SYNOPSIS
    SERVER side: signs a payload and writes the script and its manifest to the outbox.
    .DESCRIPTION
    Writes thermostat-<Version>.ps1 and thermostat-<Version>.json. The manifest holds the version, the SHA-256 of the file
    and the ECDSA P-384 / SHA-384 signature (IEEE P1363, hex).
    .PARAMETER Version
    Version of the payload, for example 1.0.0.
    .PARAMETER Script
    The text of the payload.
    .PARAMETER Dir
    Outbox folder.
    .PARAMETER SigningKey
    Private signing key (System.Security.Cryptography.ECDsa). Only the server has it.
    .NOTES
    Replaces the function that used to be defined inside notebook 06, which read the key from $signKey.
    #>
    param([Parameter(Mandatory)][string]$Version, [Parameter(Mandatory)][string]$Script, [Parameter(Mandatory)][string]$Dir,
          [Parameter(Mandatory)][Security.Cryptography.ECDsa]$SigningKey)
    $bytes = [Text.UTF8Encoding]::new($false).GetBytes($Script)
    $sig = $SigningKey.SignData(

# ---- Test-PayloadSignature


    <#
    .SYNOPSIS
    DEVICE side: checks the hash and the signature of a payload with the public key.
    .PARAMETER ScriptPath
    The payload file.
    .PARAMETER ManifestPath
    Its manifest (JSON).
    .PARAMETER PublicKeyPath
    PEM file with the public signing key (the only key the device holds).
    .OUTPUTS
    'ok', 'hash diverso' (the file was altered) or 'firma non valida' (the signature does not match).
    #>
    param([Parameter(Mandatory)][string]$ScriptPath, [Parameter(Mandatory)][string]$ManifestPath, [Parameter(Mandatory)][string]$PublicKeyPath)
    $pub = [Security.Cryptography.ECDsa]::Create()
    $pub.ImportFromPem([IO.File]::ReadAllText($PublicKeyPath))
    $bytes = [IO.File]::ReadAllBytes($ScriptPath); $m = Get-Content $ManifestPath -Raw | ConvertFrom-Json
    if ([Convert]::ToHexString([Security.Cryptography.SHA256]::HashData($bytes)).ToLower() -ne $m.sha256) { return 'hash diverso' }
    $ok = $pub.VerifyData($bytes, [Convert]::FromHexString($m.sig), [Se

# ---- Install-Payload


    <#
    .SYNOPSIS
    DEVICE side: staging, signature check, trial run, promotion; otherwise rollback to the last known good version.
    .DESCRIPTION
    Takes thermostat-<Version>.ps1 and its manifest from <Root>\server\outbox. If the signature is wrong the current payload
    does not change. Otherwise the current payload is kept as last-known-good, the new one becomes current and is run once:
    if the trial fails (no exception allowed, a temperature required) the previous version is restored.
    The device folder is <Root>\device (payload\current.ps1, payload\last-known-good.ps1, signing-public.pem).
    .PARAMETER Version
    Version to install.
    .PARAMETER Root
    Base folder of the demo (it contains server\ and device\).
    .OUTPUTS
    One line of text that says what happened.
    .NOTES
    Replaces the function that used to be defined inside notebook 06, which read $root from the notebook.
    #>
    param([Parameter(Mandatory)][string]$Version, [Parameter(Mandato

## 3 · File 1.0.0: la versione di partenza

In [ ]:
$v1 = @'
# thermostat 1.0.0
function Get-Telemetry {
    $setpoint = 21.0
    [pscustomobject]@{ payloadVersion = '1.0.0'; setpoint = $setpoint; temperature = 20.4; alarm = $false; note = 'riscalda fino a 21 C' }
}
'@
Publish-Payload -Version '1.0.0' -Script $v1 -Dir (Join-Path $root 'server\outbox') -SigningKey $signKey
Install-Payload -Version '1.0.0' -Root $root | Out-String
Get-DevicePayloadTelemetry (Join-Path $root 'device\payload\current.ps1') | Format-List | Out-String

[1.0.0] firma ok, prova superata: ATTIVO.



payloadVersion : 1.0.0
setpoint       : 21
temperature    : 20,4
alarm          : False
note           : riscalda fino a 21 C



## 4 · File 2.0.0: una nuova versione, un altro comportamento
Nuovo setpoint (19 °C) e un allarme sopra i 25 °C. L'agente è **lo stesso**: cambia solo lo script.

In [ ]:
$v2 = @'
# thermostat 2.0.0
function Get-Telemetry {
    $setpoint = 19.0; $temperature = 26.1
    [pscustomobject]@{ payloadVersion = '2.0.0'; setpoint = $setpoint; temperature = $temperature; alarm = ($temperature -gt 25); note = 'risparmio: 19 C, allarme sopra 25 C' }
}
'@
Publish-Payload -Version '2.0.0' -Script $v2 -Dir (Join-Path $root 'server\outbox') -SigningKey $signKey
Install-Payload -Version '2.0.0' -Root $root | Out-String
Get-DevicePayloadTelemetry (Join-Path $root 'device\payload\current.ps1') | Format-List | Out-String

[2.0.0] firma ok, prova superata: ATTIVO.



payloadVersion : 2.0.0
setpoint       : 19
temperature    : 26,1
alarm          : True
note           : risparmio: 19 C, allarme sopra 25 C



## 5 · Difese: manomissione, firma sbagliata, file difettoso
1. **Manomesso in transito**: si cambia un byte dopo la firma.
2. **Firmato con un'altra chiave**: chi non ha la chiave giusta non può farsi eseguire.
3. **Firmato bene ma difettoso**: la firma dice *chi* l'ha rilasciato, non che *funzioni*. Lo ferma la prova, e si torna a 2.0.0.

In [ ]:
# 1. manomesso
$v3 = $v2 -replace 'setpoint = 19.0', 'setpoint = 40.0' -replace '2.0.0', '3.0.0'
Publish-Payload -Version '3.0.0' -Script $v3 -Dir (Join-Path $root 'server\outbox') -SigningKey $signKey
Add-Content (Join-Path $root 'server\outbox\thermostat-3.0.0.ps1') "`n# aggiunto da qualcuno"
Install-Payload -Version '3.0.0' -Root $root | Out-String

# 2. firmato con un'altra chiave
$altro = [Security.Cryptography.ECDsa]::Create([Security.Cryptography.ECCurve]::CreateFromFriendlyName('nistP384'))
$bytes = [Text.UTF8Encoding]::new($false).GetBytes($v3)
$fake = $altro.SignData($bytes, [Security.Cryptography.HashAlgorithmName]::SHA384, [Security.Cryptography.DSASignatureFormat]::IeeeP1363FixedFieldConcatenation)
[IO.File]::WriteAllBytes((Join-Path $root 'server\outbox\thermostat-3.1.0.ps1'), $bytes)
[ordered]@{ version = '3.1.0'; sig = [Convert]::ToHexString($fake).ToLower(); sha256 = [Convert]::ToHexString([Security.Cryptography.SHA256]::HashData($bytes)).ToLower() } | ConvertTo-Json | Set-Content (Join-Path $root 'server\outbox\thermostat-3.1.0.json') -Encoding utf8
Install-Payload -Version '3.1.0' -Root $root | Out-String

[3.0.0] RIFIUTATO: hash diverso. Il payload corrente non cambia.


[3.1.0] RIFIUTATO: firma non valida. Il payload corrente non cambia.


In [ ]:
# 3. firmato bene ma difettoso
$v4 = @'
# thermostat 4.0.0 (difettoso)
function Get-Telemetry { throw 'sensore non inizializzato' }
'@
Publish-Payload -Version '4.0.0' -Script $v4 -Dir (Join-Path $root 'server\outbox') -SigningKey $signKey
Install-Payload -Version '4.0.0' -Root $root | Out-String
"--- il device e' ancora alla 2.0.0:"
Get-DevicePayloadTelemetry (Join-Path $root 'device\payload\current.ps1') | Format-List | Out-String

[4.0.0] firma ok ma la PROVA e' fallita (sensore non inizializzato): ritorno alla versione precedente.


--- il device e' ancora alla 2.0.0:


payloadVersion : 2.0.0
setpoint       : 19
temperature    : 26,1
alarm          : True
note           : risparmio: 19 C, allarme sopra 25 C



## Cosa abbiamo visto
- Il device **non applica mai alla cieca**: verifica hash e firma, tiene da parte la versione buona, applica, prova.
- La **firma sui byte esatti** e la chiave **solo pubblica** sul device sono il confine di sicurezza; la prova e il **ritorno alla versione precedente** coprono ciò che la firma non può dire.
- Nel notebook 14 il job **consegna** il file a un gruppo; qui il **device decide** se e come adottarlo. Le due cose si compongono: il job porta il file, il device lo verifica e lo prova.

**Fine della parte pratica** (resta il notebook 17, sui dati). Se hai eseguito 01 e 02, il resource group si elimina dall'ultima cella del notebook 99.

Prossimo notebook: **17**, Fabric: dall'Hub direttamente alla Eventstream.